# 2.15 Gradient Descent

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 2: Mathematics for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-02-mathematics-for-machine-learning/02.15_gradient-descent.ipynb)

Companion code for Section 2.15 of the book. Run the cells in order from top to bottom.

**Example 2.15.1**

In [ ]:
import numpy as np

np.random.seed(42)
# Generate linear regression data: y = 3x + 2 + noise
n = 1000
X = np.random.randn(n, 1)
y = 3 * X.squeeze() + 2 + 0.5 * np.random.randn(n)

def mse_loss(X, y, w, b):
    preds = X.squeeze()*w + b
    return np.mean((preds - y)**2)

def gradients(X, y, w, b):
    preds = X.squeeze()*w + b
    errors = preds - y
    dw = 2 * np.mean(errors * X.squeeze())
    db = 2 * np.mean(errors)
    return dw, db

# Mini-batch gradient descent
w, b    = 0.0, 0.0    # Initial parameters
lr      = 0.01
batch   = 32
epochs  = 50
history = []

for epoch in range(epochs):
    # Shuffle data each epoch
    idx = np.random.permutation(n)
    X_shuf, y_shuf = X[idx], y[idx]

    for i in range(0, n, batch):
        Xb = X_shuf[i:i+batch]
        yb = y_shuf[i:i+batch]
        dw, db = gradients(Xb, yb, w, b)
        w -= lr * dw
        b -= lr * db

    history.append(mse_loss(X, y, w, b))

print(f'True: w=3.0, b=2.0')
print(f'Fit:  w={w:.4f}, b={b:.4f}')
print(f'Initial loss: {history[0]:.4f}')
print(f'Final loss:   {history[-1]:.4f}')

**Example 2.15.2**

In [ ]:
# Learning rate comparison
lrs = {'Too large': 1.0, 'Good': 0.01, 'Too small': 0.0001}
true_w, true_b = 3.0, 2.0

for name, lr in lrs.items():
    w, b = 0.0, 0.0
    losses = []
    for epoch in range(100):
        dw, db = gradients(X, y, w, b)
        w -= lr*dw; b -= lr*db
        losses.append(mse_loss(X, y, w, b))
        if losses[-1] > 1e6: break  # diverged
    status = 'DIVERGED' if losses[-1] > 1e6 else f'loss={losses[-1]:.4f}'
    print(f'lr={lr:<8} after {len(losses):3d} epochs: {status}')

**Example 2.15.3**

In [ ]:
import torch
import torch.nn as nn

# Compare optimisers on a simple regression in PyTorch
torch.manual_seed(42)
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

def train_with_optimizer(optim_name, lr=0.01, epochs=100):
    model = nn.Linear(1, 1)  # Simple linear layer
    nn.init.zeros_(model.weight); nn.init.zeros_(model.bias)
    loss_fn = nn.MSELoss()

    if   optim_name == 'SGD':     opt = torch.optim.SGD(model.parameters(), lr=lr)
    elif optim_name == 'Momentum':opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optim_name == 'RMSProp': opt = torch.optim.RMSprop(model.parameters(), lr=lr)
    elif optim_name == 'Adam':    opt = torch.optim.Adam(model.parameters(), lr=lr)
    elif optim_name == 'AdamW':   opt = torch.optim.AdamW(model.parameters(), lr=lr)

    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X_t).squeeze(), y_t)
        loss.backward()
        opt.step()

    final_loss = loss_fn(model(X_t).squeeze(), y_t).item()
    w_fit = model.weight.item()
    b_fit = model.bias.item()
    return final_loss, w_fit, b_fit

print(f'Target: w=3.0, b=2.0')
print(f'{"Optimiser":<12} {"Final Loss":<14} {"w":<10} {"b"}')
for opt in ['SGD','Momentum','RMSProp','Adam','AdamW']:
    loss, w, b = train_with_optimizer(opt)
    print(f'{opt:<12} {loss:<14.4f} {w:<10.4f} {b:.4f}')

**Example 2.15.4**

In [ ]:
import torch

# Cosine annealing schedule - used widely in modern training
model  = torch.nn.Linear(1, 1)
optim  = torch.optim.Adam(model.parameters(), lr=0.01)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=50, eta_min=1e-5)

print(f'{"Epoch":<8} {"LR"}')
for epoch in range(1, 11):
    optim.step(); sched.step()
    print(f'{epoch:<8} {sched.get_last_lr()[0]:.6f}')
print('...')

# Warmup schedule (linear warmup then cosine decay)
# Used in BERT, GPT, and most Transformer training
def lr_lambda(step, warmup=100, total=1000):
    if step < warmup:
        return step / warmup      # Linear warmup
    progress = (step-warmup) / (total-warmup)
    return 0.5 * (1 + np.cos(np.pi * progress))  # Cosine decay

steps = [0, 50, 100, 200, 500, 900, 1000]
print('\nWarmup + Cosine schedule:')
for s in steps:
    print(f'  Step {s:4d}: lr_scale = {lr_lambda(s):.4f}')